# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and visualize the FAIR^2 dataset using the `mlcroissant` library, referencing specific entities by their `@id` values as required by the Croissant specification.

### Dataset Source
The dataset source is provided as a Croissant schema at the following URL:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\nDescription: {metadata.description}")
print(f"Identifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in the dataset are referenced by their `@id` fields.

In [ ]:
# List all record sets and fields (@id) in the dataset

# Get all record sets by their @id
record_sets = [r['@id'] for r in dataset.metadata.to_json().get('recordSet', [])]
print("Record Set @id values:")
for rs_id in record_sets:
    print(rs_id)

# For each record set, list its fields (referenced by @id)
fields_by_record_set = {}
for rs_id in record_sets:
    rs_metadata = dataset.metadata.find_by_id(rs_id)
    if rs_metadata:
        fields = [f['@id'] for f in rs_metadata.get('field', [])]
        fields_by_record_set[rs_id] = fields
        print(f"Fields for record set {rs_id}:")
        for field_id in fields:
            print(f"    {field_id}")
    else:
        print(f"No metadata found for record set {rs_id}.")

# Show a sample record from each record set
for rs_id in record_sets:
    print(f"\nSample record from record set: {rs_id}")
    try:
        record_iter = dataset.records(record_set=rs_id)
        for i, rec in enumerate(record_iter):
            print(rec)
            if i == 0:
                break
    except Exception as e:
        print(f"Error loading records for {rs_id}: {e}")

## 3. Data Extraction
Load data from the available record set(s) into a pandas DataFrame for analysis. Use record set and field `@id` values from the overview.

Note: If there is only one record set, use its @id. Extend to each record set as necessary.

In [ ]:
# Extract data from each record set into DataFrames
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Available columns (field @id): {df.columns.tolist()}")
    print(df.head(2))

# For further analysis, select the main record set, e.g., the one with the most fields/rows
if record_sets:
    main_record_set_id = record_sets[0]
else:
    main_record_set_id = None
    print('No record sets found.')

# Show a summary
if main_record_set_id:
    print(f"\nMain record set selected: {main_record_set_id}")
    print(f"DataFrame shape: {dataframes[main_record_set_id].shape}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming distributions, and grouping by key attributes.

**All field and attribute references are by their `@id`.**

In [ ]:
import numpy as np

# Inspect available numeric fields for EDA
df = dataframes[main_record_set_id]
numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
print(f"Numeric fields in record set {main_record_set_id}: {numeric_fields}")

# Select a numeric field by its @id
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = None
    print('No numeric fields found in main record set.')

# Apply EDA steps: filter, normalize, and group
if numeric_field_id:
    threshold = np.nanmean(df[numeric_field_id])
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records from {main_record_set_id} with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize numeric field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized values of {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (choose a non-numeric if available)
    group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
    if group_fields:
        group_field_id = group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped records by {group_field_id} and mean of {numeric_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric field selected for EDA.')

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. All fields must be referenced by their `@id`.

Below are basic visualizations of the selected numeric and grouping fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the numeric field
if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Visualize mean numeric field per group (if group_field_id selected)
    if 'group_field_id' in locals():
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No numeric field for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded dataset metadata and records using `mlcroissant`.
- Explored available record sets and fields by their `@id`.
- Performed basic EDA including filtering, normalization, and grouping operations based on field `@id` references.
- Visualized distributions and grouped means, illustrating the utility of Croissant schemas for transparent, reproducible data analysis.

Further exploration and modeling can proceed using the clean, structured DataFrames built directly from the Croissant schema.